In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
import ezdxf
from rasterio.plot import show
import numpy as np
from pyproj import CRS

# 0.7 - Merge RegBL and Relief

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `13_SWISSTOPO_relief`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Relief raster

In [ ]:
src = rasterio.open(f'{ROOT}data/raw/13_SWISSTOPO_relief/DHM200.asc')

relief_data = src.read(1)

print(src.profile)
print(src.bounds) 
print(src.crs)    

#### Set the corresponding CRS since it is missing

In [ ]:
with rasterio.open(f"{ROOT}data/raw/13_SWISSTOPO_relief/DHM200.asc", "r+") as src:
    src.crs = CRS.from_epsg(21781)

src = rasterio.open(f'{ROOT}data/raw/13_SWISSTOPO_relief/DHM200.asc')

relief_data = src.read(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
cax = ax.imshow(relief_data, cmap='viridis')

### Current dataset

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.4.csv', index_col=0)

df_regbl.head()

## Convert current dataset to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

## Transform current dataset to match raster's CRS (EPSG:21781)

In [ ]:
df_regbl = df_regbl.to_crs("EPSG:21781")

## Overlay relief raster on RegBL to obtain altitude for each building

In [ ]:
with rasterio.open(f'{ROOT}data/raw/13_SWISSTOPO_relief/DHM200.asc') as src:
    alt_band = src.read(1)
    raster_crs = src.crs
    transform = src.transform

    alt_values = []
    for geom in df_regbl.geometry:
        try:
            row, col = src.index(geom.x, geom.y)
            val = alt_band[row, col]
        except IndexError:
            val = None  # if point is outside raster bounds
        alt_values.append(val)

df_regbl["Relief - Altitude (m)"] = alt_values

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
df_regbl.plot(column='Relief - Altitude (m)', ax=ax, legend=True, cmap='viridis')
plt.show()

## Transform current dataset CRS back to EPSG:2056

In [ ]:
df_regbl = df_regbl.to_crs("EPSG:2056")

## Convert current dataset back to DataFrame

In [ ]:
df_regbl = pd.DataFrame(df_regbl.drop(columns='geometry'))

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.5.csv")